# logit-thermo 探索：文章の DSC その 3 ——高温高圧（強い ＋h）で、凍る・溶けるの境目の線はどうなるか（臨界点はあるか）の下見（Pythia-1B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 1 時間 10〜20 分。終わったら自動で切れます。
- もう 1 本（OLMo-2-1B）のノートブックと、別のランタイムで同時に回せます（保存先のフォルダがモデルごとに別です）。
- 探索（下見、予想なし）。場 h（圧力にあたる）を h ＝ 0・＋0.5・＋1.0・＋2.0 の 4 本の腕にして、どの腕も 2.0 → 0.35 を 800 歩で冷やし、0.35 → 2.0 を 800 歩で温めます。記事 40 × サンプル 4。各歩の q（候補の確率の和）とエネルギー U も記録します。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコード、`scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）、`scripts/a2_explore_dsc.py`（`33f47a0a3bbda183…`）、`scripts/a2_explore_dsc2.py`（`cddae78690360418…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_explore_dsc3.py`（`e650939ff85a161f…`）はこのノートブックに埋め込み。
- 保存先 `A2_explore/dsc3_pythia/`。腕は 4 本。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。**結果は表示しません。**
- **通知**：🚀・腕ごとの ✅（4 回）・🎉・⚠️。名札は「logit-thermo 探索 文章の DSC その 3（高温高圧 h 0〜＋2、2.0 まで、Pythia-1B、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 文章の DSC その 3（高温高圧 h 0〜＋2、2.0 まで、Pythia-1B、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore DSC3 high T high h Pythia-1B', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCD3 = '"""探索 8.39（下見）：文章の DSC その 3 ——高温高圧（強い ＋h）で、凍る・溶けるの境目の線がどうなるか（臨界点はあるか）（2026-10-31 18:1x、GPU（L4）、判定しない）。\nscripts/a2_2_collect.py・a2_4_collect.py・a2_7_collect.py・a2_7b_collect.py・a2_explore_field.py・a2_explore_dsc.py・a2_explore_dsc2.py を実行したあとの\n同じ名前空間で実行する（load_model・valid_mask・split_doc・D_MODELS・D_CHUNK・generate_dsc2 を変えずに使う）。\n\nh は圧力にあたる（＋h で融点が上がる、8.30）。腕は h ＝ 0・＋0.5・＋1.0・＋2.0（はじめから終わりまで同じ h）。\n温度：2.0 → 0.35 を 800 歩で冷やし、0.35 → 2.0 を 800 歩で温める（1 本 1,600 歩）。上の端を 2.0 にしたのは、h ＝ ＋2 で溶ける温度が 1.5 を超えうるため\n（8.38 の 1.13 ＋ 傾き 0.27 × 2 ≈ 1.67）。温める速さ（1 歩 0.0021）は 8.38 の base（0.0018）に近い。\n種は腕によらず同じ（D3_SEED ＋ 10 s ＋ c、共通の乱数。場がはじめからかかるので、トークンは腕ごとに違う）。\n記録は 8.38 と同じ：トークン、候補があったか、候補を選んだか、U（温度 1・場なし）、q0（場の前）、q（場の後）。\n"""\nimport gc, os, time\nimport numpy as np\nimport torch\n\nD3_TOP, D3_BOT, D3_N = 2.0, 0.35, 800\nD3_ARMS = {\'h0\': 0.0, \'hp05\': 0.5, \'hp10\': 1.0, \'hp20\': 2.0}\nD3_S = 4\nD3_SEED = 20261139\n\n\ndef d3_sched(arm):\n    Ts = np.r_[np.linspace(D3_TOP, D3_BOT, D3_N), np.linspace(D3_BOT, D3_TOP, D3_N)]\n    return Ts, np.full_like(Ts, D3_ARMS[arm])\n\n\ndef d3_seed(s, c):\n    return D3_SEED + 10 * s + c\n\n\ndef run_dsc3(name, hf_id, docs, outdir, device, notify=print):\n    todo = [a for a in D3_ARMS if not os.path.exists(f\'{outdir}/{name}@{a}.npz\')]\n    if not todo:\n        print(name, \'はもう全部終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nD = len(use)\n    P = {len(pairs[i][0]) for i in use}\n    assert len(P) == 1\n    prompts = [pairs[i][0] for i in use]\n    maxpos = getattr(model.config, \'max_position_embeddings\', None)\n    assert maxpos is None or P.pop() + 2 * D3_N <= maxpos, (\'文が長すぎます\', maxpos)\n    chunks = [list(range(i, min(nD, i + D_CHUNK))) for i in range(0, nD, D_CHUNK)]\n    for arm in todo:\n        ta = time.time()\n        Ts, Hs = d3_sched(arm)\n        L = len(Ts)\n        dt = dict(toks=np.int32, cand=np.bool_, pick=np.bool_, U=np.float32, q0=np.float32, q=np.float32)\n        R = {k: np.zeros((D3_S, nD, L), v) for k, v in dt.items()}\n        for s in range(D3_S):\n            for c, ii in enumerate(chunks):\n                g = torch.Generator(device=device).manual_seed(d3_seed(s, c))\n                r = generate_dsc2(model, [prompts[i] for i in ii], Ts, Hs, valid, g)\n                for k in R:\n                    R[k][s, ii] = r[k]\n                torch.cuda.empty_cache()\n        path = f\'{outdir}/{name}@{arm}.npz\'\n        np.savez_compressed(path + \'.part.npz\', Ts=Ts, Hs=Hs, arm=arm, use=np.array(use), seconds=time.time() - ta, chunk=D_CHUNK, **R)\n        os.replace(path + \'.part.npz\', path)\n        notify(f\'{name}@{arm} おわり（{time.time() - ta:.0f} 秒、通算 {time.time() - t0:.0f} 秒）\')\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\ndef self_test_dsc3(device):\n    """予定（長さ・温度の端・場）と、小さなランダムのモデルで h ＝ ＋2 の q ≥ q0（候補がある歩）。"""\n    for arm, h in D3_ARMS.items():\n        Ts, Hs = d3_sched(arm)\n        assert len(Ts) == 1600 and abs(Ts[0] - 2.0) < 1e-12 and abs(Ts[799] - 0.35) < 1e-12 and abs(Ts[800] - 0.35) < 1e-12 and abs(Ts[-1] - 2.0) < 1e-12\n        assert (Hs == h).all()\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4]\n    Ts = np.linspace(2.0, 0.35, 60)\n    b = generate_dsc2(model, prompts, Ts, np.full(60, 2.0), valid, torch.Generator(device=device).manual_seed(9))\n    m = b[\'cand\']\n    assert (b[\'q\'][m] >= b[\'q0\'][m] - 1e-6).all() and (b[\'q\'][~m] == 0).all()\n    return \'OK\'\n'
assert hashlib.sha256(SRCD3.encode()).hexdigest() == 'e650939ff85a161f13572a3fd837baf5206e1726dd14a9a2ae2c4260c6a7bfcf'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCD3)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 4 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'), self_test_dsc3('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore/dsc3_pythia', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = D_MODELS['pythia']
    run_dsc3(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_explore/dsc3_pythia', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(model=[name, hf_id], arms=D3_ARMS, top=D3_TOP, bot=D3_BOT, n=D3_N, samples=D3_S, chunk=D_CHUNK, seed=D3_SEED,
                   dsc2_sha256='cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', dsc3_sha256='e650939ff85a161f13572a3fd837baf5206e1726dd14a9a2ae2c4260c6a7bfcf', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_explore/dsc3_pythia' + f'/run_meta_{name}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 文章の DSC その 3（Pythia-1B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：文章の DSC その 3（Pythia-1B）の材料を A2_explore/dsc3_pythia に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
